<a href="https://colab.research.google.com/github/Dhrupad-05/Projects/blob/main/Sentiment_Analysis_of_Movie_Reviews_using_SVM%2C_Logistic_Regression%2C_and_Naive_Bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentiment Analysis of Movie Reviews using SVM, Logistic Regression, and Naive Bayes

**Overview**

In [2]:
#IMport Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# NLP and preprocessing
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import nltk

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# ML models
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

# Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

import os
import glob

print(" All libraries imported successfully")

 All libraries imported successfully


##  Dataset Overview and Loading

In [9]:
import tarfile
import os

file_path = "/content/aclImdb_v1 (1).tar.gz"

print(f"1. File exists: {os.path.exists(file_path)}")
print(f"2. File size: {os.path.getsize(file_path) / 1024 / 1024:.1f} MB")

try:
    with tarfile.open(file_path, 'r:gz') as tar:
        print(f"3. ✅ File is valid tar.gz")
        print(f"4. Contains {len(tar.getmembers())} files")
except Exception as e:
    print(f"3. ❌ File is corrupted: {str(e)[:100]}")
    print(f"4. Download fresh copy using FASTEST FIX above")

1. File exists: True
2. File size: 80.2 MB
3. ✅ File is valid tar.gz
4. Contains 100019 files


In [10]:
# Function to load data from directory structure
def load_imdb_data(base_path, max_samples=5000):
    texts = []
    labels = []

    # Load positive reviews
    pos_files = glob.glob(os.path.join(base_path, 'pos/*.txt'))[:max_samples//2]
    for file_path in pos_files:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            texts.append(f.read())
            labels.append(1)  # Positive

    # Load negative reviews
    neg_files = glob.glob(os.path.join(base_path, 'neg/*.txt'))[:max_samples//2]
    for file_path in neg_files:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            texts.append(f.read())
            labels.append(0)  # Negative

    return texts, labels

# Load training data
train_path = '/tmp/aclImdb/train'
texts, labels = load_imdb_data(train_path, max_samples=5000)

# Create DataFrame
df = pd.DataFrame({'review': texts, 'sentiment': labels})

print(f"Dataset Shape: {df.shape}")
print(f"\nFirst 3 rows:")
print(df.head(3))
print(f"\nData Types:\n{df.dtypes}")
print(f"\nMissing Values:\n{df.isnull().sum()}")

Dataset Shape: (5000, 2)

First 3 rows:
                                              review  sentiment
0  Rohinton Mistry's multi-layered novel seemed i...          1
1  The movie "MacArthur" begins and ends at Gen. ...          1
2  First of all, when people hear 'GUY RITCHIE', ...          1

Data Types:
review       object
sentiment     int64
dtype: object

Missing Values:
review       0
sentiment    0
dtype: int64


### Dataset Description

**IMDB Movie Reviews Dataset:**
- **Source:** Large Movie Review Dataset (IMDB)
- **Samples:** 5,000 reviews (2,500 positive, 2,500 negative)
- **Label Distribution:** Binary classification (Positive=1, Negative=0)
- **Review Length:** Variable (short to very long)

**Columns:**
- `review`: Text of the movie review
- `sentiment`: Label (1=Positive, 0=Negative)